# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [12]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [13]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    !cd ECE1508_GenAI && git pull

%cd ECE1508_GenAI

Cloning into 'ECE1508_GenAI'...
remote: Enumerating objects: 1004, done.
remote: Counting objects: 100% (516/516), done.
remote: Compressing objects: 100% (390/390), done.
remote: Total 1004 (delta 248), reused 388 (delta 126), pack-reused 488 (from 1)
Receiving objects: 100% (1004/1004), 37.33 MiB | 44.55 MiB/s, done.
Resolving deltas: 100% (449/449), done.
/content/ECE1508_GenAI/ECE1508_GenAI


In [14]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

In [15]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [16]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI/ECE1508_GenAI
plugins: anyio-4.14.2, typeguard-4.5.2, langsmith-0.10.2
collected 34 items                                                             

steven/tests/test_data_pipeline.py::test_reconstruct_prices_round_trip PASSED [  2%]
steven/tests/test_data_pipeline.py::test_anchor_correction_matches_close_0_for_all_horizon_bars PASSED [  5%]
steven/tests/test_data_pipeline.py::test_wick_components_non_negative PASSED [  8%]
steven/tests/test_data_pipeline.py::test_build_window_shapes_and_masks PASSED [ 11%]
steven/tests/test_data_pipeline.py::test_to_patchtst_input_patch_padding_mask PASSED [ 14%]
steven/tests/test_data_pipeline.py::test_window_sampler_unique_and_within_bounds PASSED [ 17%]
steven/tests/test_data_pipeline.py::test_window_sampler_respects_split_boundary

## Train PatchTST (real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first instead of the full config.

In [17]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

19:09:38 device: cuda
19:09:38 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
19:09:38 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
19:09:42 epoch 1/20  train_loss=0.23157  val_loss=0.10953  (2.4s)
19:09:42   -> saved best checkpoint (val_loss=0.10953) to steven/outputs/patchtst_checkpoint.pt
19:09:44 epoch 2/20  train_loss=0.16731  val_loss=0.11391  (1.9s)
19:09:45 epoch 3/20  train_loss=0.15137  val_loss=0.09968  (1.9s)
19:09:45   -> saved best checkpoint (val_loss=0.09968) to steven/outputs/patchtst_checkpoint.pt
19:09:47 epoch 4/20  train_loss=0.14287  val_loss=0.09491  (2.0s)
19:09:47   -> saved best checkpoint (val_loss=0.09491) to steven/outputs/patchtst_checkpoint.pt
19:09:49 epoch 5/20  train_loss=0.14188  val_loss=0.09595  (1.9s)
19:09:51 epoch 6/20  train_loss=0.13820  val_loss=0.09251  (1.9s)
19:09:51   -> saved best checkpoint (val_loss=0.09251) to steven/outputs/patchtst_checkpoint.pt
19:09:53 epoch 7/20  train_

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [18]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

19:10:21 device: cuda
19:10:21 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
19:10:21 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
19:10:24 epoch 1/30  beta=0.20  train_loss=0.61205 (kl=0.8034)  val_loss=0.39194 (kl=0.8000)  (2.0s)
19:10:24   -> saved best checkpoint (val_loss=0.39194) to steven/outputs/cvae_checkpoint.pt
19:10:25 epoch 2/30  beta=0.40  train_loss=0.62767 (kl=0.8001)  val_loss=0.51638 (kl=0.8000)  (1.4s)
19:10:27 epoch 3/30  beta=0.60  train_loss=0.76279 (kl=0.8001)  val_loss=0.65916 (kl=0.8000)  (1.4s)
19:10:28 epoch 4/30  beta=0.80  train_loss=0.88800 (kl=0.8002)  val_loss=0.81228 (kl=0.8001)  (1.4s)
19:10:30 epoch 5/30  beta=1.00  train_loss=1.00540 (kl=0.8002)  val_loss=0.95121 (kl=0.8000)  (1.4s)
19:10:31 epoch 6/30  beta=1.00  train_loss=0.98621 (kl=0.8000)  val_loss=0.93266 (kl=0.8000)  (1.4s)
19:10:32 epoch 7/30  beta=1.00  train_loss=0.97843 (kl=0.8000)  val_loss=0.92919 (kl=0.8000)  (1.4s)
19:10:34

## Evaluate both models on the fixed test set

In [19]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

19:11:09 device: cuda
19:11:09 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
19:11:09 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
19:11:09 sell-price shrink bound: p99.0 of |anchored log return| over train = 0.0190 (vs. model's own MAX_LOG_RETURN)
19:11:09 evaluating on 3000 fixed test windows
19:11:10 wrote metrics to steven/outputs/metrics.json
19:11:10 overall: {
  "n_windows": 3000,
  "patchtst_reparam_mae_rmse": [
    0.09608245640993118,
    0.3724913001060486
  ],
  "cvae_reparam_mae_rmse": [
    0.16340036690235138,
    0.49513185024261475
  ],
  "patchtst_ohlc_mae_rmse": [
    2.308971835489268,
    3.4452575587323877
  ],
  "cvae_ohlc_mae_rmse": [
    6.876354131122346,
    8.527548357618938
  ],
  "patchtst_volume_mae_rmse": [
    1944256.0,
    3501149.25
  ],
  "cvae_volume_mae_rmse": [
    3227552.75,
    4625670.5
  ],
  "patchtst_directional_accuracy": [
    0.526,
    0.524,
    0.5416666666666666
  ],
  "c

## Refresh v1.md from this run

Rewrites the Results/backtest tables and sample images in `steven/v1.md` from the metrics.json + sample_plots this run just produced (see `steven/src/update_report.py`). Only the tables/images are rewritten -- surrounding prose (interpretation, caveats) is left as-is; review it by hand if the story changed. This only edits the file in the cloned repo here -- push/download separately if you want to keep it.

In [20]:
!python steven/src/update_report.py

19:11:14 updated steven/v1.md: results-samples, hit-summary, spread-summary, outcome-breakdown, backtest-patchtst, backtest-cvae, buy-hold-benchmark
19:11:14 not auto-updated -- reread and edit by hand if the story changed: the 'In plain terms' / 'A subtle but important point' interpretation paragraphs under Results, the 'pre-retrain checkpoints' caveats in Results and Long-only backtest results, and the 'Retrain both models' checkbox under Next steps.


## Sync results back to GitHub

Commits `steven/outputs/` (checkpoints, metrics.json, sample_plots) and the regenerated `steven/v1.md` from this Colab runtime and pushes straight to the `steven` branch -- no manual zip/download step. That step wasn't reliably reaching the local machine: `files.download()`'s browser-download trick only works from the Colab web UI, not when this kernel is attached remotely (e.g. from VS Code's kernel picker), so nothing ever landed on disk.

Needs a GitHub personal access token with `repo` write scope for this push only -- entered via `getpass` below, never written to the notebook or committed anywhere.

In [21]:
import getpass

token = getpass.getpass("GitHub PAT (repo write, used only for this push): ")

In [ ]:
%%bash -s "$token"
TOKEN="$1"
if [ -z "$TOKEN" ]; then
  echo "Token was empty -- re-run the getpass cell above and actually paste your PAT before pressing Enter." >&2
  exit 1
fi
git config user.email "colab@ephemeral.local"
git config user.name "Colab Runtime"
git add steven/outputs steven/v1.md
if git diff --cached --quiet; then
  echo "Nothing new to commit -- outputs/v1.md unchanged from last commit."
else
  git commit -m "Retrain + refresh results from Colab run"
fi
# Push unconditionally -- a prior run may have committed but failed to push (e.g. a blank
# token), in which case there's nothing new to commit here but HEAD is still ahead of origin.
git push "https://${TOKEN}@github.com/WoodyChang21/ECE1508_GenAI.git" HEAD:steven

### Fallback: zip + browser download

Only useful if you're running this notebook inside the actual Colab web UI (not a remote kernel) and would rather download a zip than push through git.

In [ ]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")